## Step 9b - Fast captions with Florence-2-large (caption ONLY, no OCR)

Alternative to step 9 when GPU-hours are the bottleneck. **Florence-2-large
(0.77B)** is a dedicated captioner: with `<MORE_DETAILED_CAPTION>` + batch 32 it
runs ~0.1-0.2 s/frame on a T4 - the full ~244k keyframes fit in ONE Kaggle
session on T4 x2 (videos are sharded across both GPUs by subprocess).

Trade-offs vs step 9 (Qwen2.5-VL):
- English captions only - fine for retrieval: the query planner already emits
  English variants and multilingual-e5 matches cross-lingually.
- NO Vietnamese OCR - keep your existing `ocr_data.json`, this step only
  produces `caption_data.json`.

**Outputs** (resumable per video AND per frame, same layout as step 9):

```
captions_florence/<video_name>.json  # { "<frame_id>": "<caption>" }
caption_data.json                    # merged -> backend data/captions/
```

Set **Accelerator = GPU T4 x2**.

In [ ]:
# Florence-2 remote code: cần einops + timm; flash_attn KHÔNG cài được trên T4
# (worker vá get_imports để bỏ qua import đó và chạy attention sdpa).
!pip install -q -U "transformers==4.49.0" einops timm pillow

In [ ]:
import os

# Step 3 keyframes (nested: Keyframes/<batch>/<video_name>/<idx>.webp)
KEYFRAMES_DIR = '/kaggle/input/aic-keyframes-webp/Keyframes'
OUTPUT_DIR    = './captions_florence'   # one <video_name>.json per video
FRAME_STRIDE  = 1                       # 1 = mọi keyframe; 2 = nửa chi phí
BATCH_SIZE    = 32                      # T4 16GB chịu được 32 với fp16

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
%%writefile worker_florence.py
"""Caption worker: một process / một GPU, nhận shard video theo chỉ số chẵn lẻ.

Chạy bằng: CUDA_VISIBLE_DEVICES=<gpu> python worker_florence.py \
    <shard> <n_shards> <keyframes_dir> <output_dir> <frame_stride> <batch_size>
"""
import json
import sys
import time
from pathlib import Path
from unittest.mock import patch

import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor
from transformers.dynamic_module_utils import get_imports

SHARD, N_SHARDS = int(sys.argv[1]), int(sys.argv[2])
KEYFRAMES_DIR, OUTPUT_DIR = sys.argv[3], sys.argv[4]
FRAME_STRIDE, BATCH_SIZE = int(sys.argv[5]), int(sys.argv[6])

MODEL_ID = 'microsoft/Florence-2-large'
TASK = '<MORE_DETAILED_CAPTION>'
MAX_NEW_TOKENS = 128
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.webp')


def _no_flash_attn(filename):
    # Florence remote code khai báo flash_attn trong imports dù không bắt buộc;
    # T4 (sm75) không cài được flash_attn -> lọc bỏ rồi chạy sdpa.
    return [i for i in get_imports(filename) if i != 'flash_attn']


with patch('transformers.dynamic_module_utils.get_imports', _no_flash_attn):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, trust_remote_code=True,
        attn_implementation='sdpa',
    ).to('cuda').eval()
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f'[shard {SHARD}] model ready', flush=True)


@torch.inference_mode()
def caption_batch(paths):
    imgs = [Image.open(p).convert('RGB') for p in paths]
    inputs = processor(text=[TASK] * len(imgs), images=imgs,
                       return_tensors='pt').to('cuda', torch.float16)
    ids = model.generate(input_ids=inputs['input_ids'],
                         pixel_values=inputs['pixel_values'],
                         max_new_tokens=MAX_NEW_TOKENS,
                         num_beams=1, do_sample=False)
    texts = processor.batch_decode(ids, skip_special_tokens=False)
    caps = []
    for t, im in zip(texts, imgs):
        d = processor.post_process_generation(t, task=TASK,
                                              image_size=(im.width, im.height))
        caps.append(str(d.get(TASK, '')).strip())
    return caps


video_dirs = sorted(d for d in Path(KEYFRAMES_DIR).glob('*/*') if d.is_dir())
video_dirs = [d for i, d in enumerate(video_dirs) if i % N_SHARDS == SHARD]
print(f'[shard {SHARD}] {len(video_dirs)} videos | stride={FRAME_STRIDE}', flush=True)

t0, n_done = time.time(), 0
for vi, video_dir in enumerate(video_dirs, 1):
    video_name = video_dir.name
    out_path = Path(OUTPUT_DIR) / f'{video_name}.json'
    done = json.loads(out_path.read_text(encoding='utf-8')) if out_path.exists() else {}
    frames = sorted(p for p in video_dir.iterdir()
                    if p.suffix.lower() in IMAGE_EXTS)[::FRAME_STRIDE]
    todo = [(f'{video_name}_{p.stem}', p) for p in frames
            if f'{video_name}_{p.stem}' not in done]
    for b in range(0, len(todo), BATCH_SIZE):
        batch = todo[b:b + BATCH_SIZE]
        try:
            caps = caption_batch([p for _, p in batch])
        except Exception as e:            # một batch hỏng không dừng cả shard
            print(f'[WARN shard {SHARD}] {video_name}: {type(e).__name__}: {e}',
                  flush=True)
            caps = [''] * len(batch)
        for (frame_id, _), c in zip(batch, caps):
            done[frame_id] = c
        out_path.write_text(json.dumps(done, ensure_ascii=False), encoding='utf-8')
    n_done += len(todo)
    rate = n_done / max(time.time() - t0, 1e-9)
    print(f'[shard {SHARD}] {vi}/{len(video_dirs)} {video_name}: '
          f'+{len(todo)} frames | {rate:.1f} f/s', flush=True)

print(f'[shard {SHARD}] DONE {n_done} frames in {(time.time() - t0) / 3600:.2f} h',
      flush=True)

In [ ]:
# Mỗi GPU một worker process, chia video theo chỉ số chẵn/lẻ. Resume: chạy lại
# cell là tiếp tục - video đủ frame bị bỏ qua ngay trong worker.
import os
import subprocess
import sys
import time

import torch

N_GPU = max(torch.cuda.device_count(), 1)
procs = []
for shard in range(N_GPU):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(shard))
    log = open(f'shard{shard}.log', 'a')
    procs.append(subprocess.Popen(
        [sys.executable, 'worker_florence.py', str(shard), str(N_GPU),
         KEYFRAMES_DIR, OUTPUT_DIR, str(FRAME_STRIDE), str(BATCH_SIZE)],
        env=env, stdout=log, stderr=subprocess.STDOUT))
print(f'{N_GPU} workers started')

while any(p.poll() is None for p in procs):
    time.sleep(120)
    for shard in range(N_GPU):
        lines = open(f'shard{shard}.log').read().strip().splitlines()
        print(lines[-1] if lines else f'[shard {shard}] no output yet')
for p in procs:
    print('exit code:', p.wait())

In [ ]:
# Merge -> caption_data.json (format backend data/captions/)
import json
from pathlib import Path

cap_merged = {}
for f in sorted(Path(OUTPUT_DIR).glob('*.json')):
    for frame_id, cap in json.loads(f.read_text(encoding='utf-8')).items():
        if cap:
            cap_merged[frame_id] = cap

with open('caption_data.json', 'w', encoding='utf-8') as fh:
    json.dump(cap_merged, fh, ensure_ascii=False)
print('caption_data.json:', len(cap_merged), 'frames')

## Use with the backend

```text
caption_data.json -> backend/data/captions/caption_data.json
```

Rồi build index caption (OCR/ASR không đổi nên chỉ thiếu caption index):

```bash
cd data_processing
python 07_build_text_indexes.py    # tự build caption_text.index khi thấy captions/
```

Backend tự nhận: `/search_caption` + kênh `caption` trong agent fan-out hoạt
động khi `caption_text.index` tồn tại. Caption tiếng Anh - multilingual-e5
match được query tiếng Việt, và planner cũng đã sinh keyword tiếng Anh.